# Experimento MIMIC-IV: validación temporal del modelo de reingreso a 30 días
**Lucas Didone · Segunda iteración · 8 de octubre de 2026**  
Fundamentos de Medicina y Aplicaciones de Inteligencia Artificial

**Por qué este notebook.** El dataset de UCI no tiene fechas, por lo que no permite entrenar en un período y evaluar en otro posterior. MIMIC-IV (Beth Israel Deaconess Medical Center, Boston) tiene admisiones de 2008 a 2022 y la variable `anchor_year_group`, que ubica a cada paciente en un trienio real. Esto permite la **validación temporal** de la acción 2 de la devolución: entrenar en los períodos iniciales, validar en el siguiente y evaluar fuera de tiempo (OOT), midiendo cuánto se degrada el desempeño.

**Diseño fijado antes de ver resultados:**

| Partición | `anchor_year_group` | Uso |
|---|---|---|
| Train | 2008–2010, 2011–2013 | Ajuste |
| Validación | 2014–2016 | Selección final, recalibración, umbral del cupo 20 % |
| OOT | 2017–2019 | **Evaluación principal**, modelo congelado |
| OOT 2 | 2020–2022 (si existe en la versión descargada) | Secundaria: incluye la pandemia de COVID-19 |

- **Validación cruzada temporal con ventana expansiva** dentro de desarrollo: (1) train 2008–10 → evaluar 2011–13; (2) train 2008–13 → evaluar 2014–16. La misma grilla modelo × desbalance que en UCI y la misma regla de un error estándar.
- Cada paciente pertenece a un único `anchor_year_group`, así que ninguna persona cruza particiones.
- **Cohorte:** adultos (≥18 años; MIMIC tiene edad exacta) con diagnóstico de diabetes (CIE-9 250.x o CIE-10 E08–E13), vivos al alta, sin alta a hospice.
- **Outcome:** próxima admisión del mismo paciente dentro de 30 días del alta (cualquier causa, solo dentro del BIDMC). Sensibilidad: excluir reingresos electivos.
- **LACE y HOSPITAL completos:** MIMIC tiene hemoglobina, sodio, servicio de alta (oncología) y visitas a guardia.
- **Transición CIE-9 → CIE-10 (octubre de 2015):** cae en el período de validación. Para que el modelo no dependa de la versión de codificación, los diagnósticos entran solo como grupos por capítulo y comorbilidades de Charlson, definidos para ambas versiones.

**Datos requeridos (acceso con credencial).** Ver la celda siguiente: si faltan los archivos, el notebook se detiene con las instrucciones. Los datos de MIMIC no se versionan (`data/mimic-iv/` está en `.gitignore`), según el acuerdo de uso de datos (DUA).

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '4')
os.environ.setdefault('MLFLOW_DISABLE_AGENT_HINT', '1')
os.environ.setdefault('MLFLOW_LOGGING_LEVEL', 'WARNING')
from pathlib import Path
import sys, json, time, platform, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
BASE = Path.cwd()
sys.path.insert(0, str(BASE))
MIMIC_DIR = Path(os.environ.get('MIMIC_IV_DIR', BASE / 'data' / 'mimic-iv'))
HOSP, ED = MIMIC_DIR / 'hosp', MIMIC_DIR / 'ed'
REQUIRED = ['patients', 'admissions', 'diagnoses_icd', 'procedures_icd', 'labevents',
            'prescriptions', 'services']

def find_table(folder, name):
    for ext in ('.csv.gz', '.csv', '.parquet'):
        p = folder / f'{name}{ext}'
        if p.exists():
            return p
    return None

class StopExecution(Exception):
    def _render_traceback_(self):
        return []

missing = [t for t in REQUIRED if find_table(HOSP, t) is None]
if missing:
    print(f'''No se encontraron datos de MIMIC-IV en {HOSP}
Faltan: {", ".join(missing)}

Pasos para obtener acceso (PhysioNet):
  1. Crear una cuenta en https://physionet.org/register/
  2. Completar el curso CITI "Data or Specimens Only Research" y subir el certificado
     (https://physionet.org/about/citi-course/).
  3. Solicitar acceso con credencial (requiere una referencia, por ejemplo el director de la carrera)
     y firmar el DUA de MIMIC-IV en https://physionet.org/content/mimiciv/
  4. Descargar el módulo hosp (y opcionalmente ed, de MIMIC-IV-ED) y copiar los archivos a:
       {HOSP}/  -> {", ".join(t + ".csv.gz" for t in REQUIRED)}
       {ED}/    -> edstays.csv.gz (opcional, mejora el componente E de LACE)
     También se puede apuntar a otra carpeta con la variable de entorno MIMIC_IV_DIR.
No compartir ni versionar estos archivos.''')
    raise StopExecution
print('Datos encontrados en', MIMIC_DIR)

In [ ]:
import duckdb, sklearn, xgboost, lightgbm, imblearn, dcurves
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.calibration import calibration_curve
from threadpoolctl import threadpool_limits
import reingresos_utils as ru
from reingresos_utils import metrics, top_metrics
threadpool_limits(limits=4)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
logging.getLogger('mlflow').setLevel(logging.WARNING)
SEED = ru.SEED
N_JOBS = min(6, os.cpu_count() or 2)
OUT = BASE / 'output' / 'mimic'; OUT.mkdir(parents=True, exist_ok=True)
CACHE = MIMIC_DIR / 'cache'; CACHE.mkdir(exist_ok=True)
MLRUNS = BASE / 'mlruns'
plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False, 'font.size': 11})
versions = {'python': platform.python_version(), 'duckdb': duckdb.__version__, 'scikit-learn': sklearn.__version__,
            'xgboost': xgboost.__version__, 'lightgbm': lightgbm.__version__, 'imbalanced-learn': imblearn.__version__}
print(versions)
con = duckdb.connect()
con.execute("SET threads TO 4")

def src(name, folder=HOSP, types=None, all_varchar=False):
    p = find_table(folder, name)
    if p.suffix == '.parquet':
        return f"read_parquet('{p.as_posix()}')"
    opts = ['header=true']
    if types:
        opts.append('types={' + ', '.join(f"'{k}': '{v}'" for k, v in types.items()) + '}')
    if all_varchar:
        opts.append('all_varchar=true')
    return f"read_csv('{p.as_posix()}', {', '.join(opts)})"

TS = 'TIMESTAMP'

## 1. Cohorte y outcome
El reingreso se busca entre **todas** las admisiones del paciente (no solo las de diabetes). Se cuentan las exclusiones en orden.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TABLE adm AS
SELECT a.subject_id, a.hadm_id, a.admittime, a.dischtime, a.edregtime, a.admission_type,
       a.admission_location, a.discharge_location, a.hospital_expire_flag,
       p.gender, p.anchor_age, p.anchor_year, p.anchor_year_group,
       LEAD(a.admittime) OVER w AS next_admittime, LEAD(a.admission_type) OVER w AS next_type
FROM {src('admissions', types={'admittime': TS, 'dischtime': TS, 'edregtime': TS, 'edouttime': TS, 'deathtime': TS})} a
JOIN {src('patients', types={'dod': 'DATE'})} p USING (subject_id)
WINDOW w AS (PARTITION BY a.subject_id ORDER BY a.admittime)''')
con.execute(f'''
CREATE OR REPLACE TABLE dx AS
SELECT subject_id, hadm_id, seq_num, icd_code, icd_version
FROM {src('diagnoses_icd', types={'icd_code': 'VARCHAR', 'icd_version': 'INTEGER'})}''')
adm = con.execute('''
SELECT a.*, a.anchor_age + (year(a.admittime) - a.anchor_year) AS age,
       a.hadm_id IN (SELECT hadm_id FROM dx WHERE (icd_version = 9 AND icd_code LIKE '250%')
                     OR (icd_version = 10 AND substr(icd_code, 1, 3) IN ('E08','E09','E10','E11','E13'))) AS diabetes
FROM adm a''').df()
cohort = adm.copy()
flow = [{'etapa': 'Admisiones MIMIC-IV', 'episodios': len(cohort), 'excluidos': 0}]
for name, keep in [
    ('Edad ≥18 años', lambda d: d.age >= 18),
    ('Diagnóstico de diabetes', lambda d: d.diabetes),
    ('Vivo al alta', lambda d: (d.hospital_expire_flag == 0) & d.discharge_location.ne('DIED')),
    ('Sin alta a hospice', lambda d: d.discharge_location.ne('HOSPICE')),
    ('Fechas válidas', lambda d: d.dischtime > d.admittime)]:
    n0 = len(cohort)
    cohort = cohort.loc[keep(cohort)].copy()
    flow.append({'etapa': name, 'episodios': len(cohort), 'excluidos': n0 - len(cohort)})
cohort = cohort.sort_values(['subject_id', 'admittime']).reset_index(drop=True)
gap = (cohort.next_admittime - cohort.dischtime).dt.total_seconds() / 86400
ELECTIVE = ['ELECTIVE', 'SURGICAL SAME DAY ADMISSION']
cohort['y'] = ((gap > 0) & (gap <= 30)).astype(int)
cohort['y_no_elective'] = (cohort.y.eq(1) & ~cohort.next_type.isin(ELECTIVE)).astype(int)
cohort['los_days'] = (cohort.dischtime - cohort.admittime).dt.total_seconds() / 86400
cohort['period'] = cohort.anchor_year_group.str.replace(' ', '')
display(pd.DataFrame(flow))
print(f'Cohorte: {len(cohort):,} internaciones de {cohort.subject_id.nunique():,} pacientes; '
      f'prevalencia {cohort.y.mean():.2%} (sin electivos: {cohort.y_no_elective.mean():.2%})')

## 2. Features al alta
Todas se calculan con información de la internación índice o anterior a su ingreso. Raza, seguro y estado civil quedan fuera del predictor, como en UCI.

- **Utilización previa:** internaciones en 6 y 12 meses previos; visitas a guardia en 6 meses (de `ed/edstays` si existe; si no, admisiones que pasaron por guardia).
- **Diagnósticos:** cantidad, grupo del diagnóstico principal (agrupación de Strack, equivalente CIE-9/CIE-10), 17 comorbilidades de Charlson y su puntaje.
- **Procedimientos:** cantidad de procedimientos CIE codificados.
- **Laboratorio:** último valor de hemoglobina, sodio y glucemia durante la internación; HbA1c categorizada como en UCI (`None` = no medida, `Norm` < 7 %, `>7` = 7–8 %, `>8`). La lectura de `labevents` se guarda en caché.
- **Medicación:** cantidad de fármacos distintos, cantidad de antidiabéticos, insulina y metformina.
- **Servicio al alta** (`services`) y destino al alta.

In [ ]:
con.register('cohort_df', cohort[['subject_id', 'hadm_id', 'admittime', 'dischtime']])
con.execute('CREATE OR REPLACE TABLE c AS SELECT * FROM cohort_df')
util = con.execute('''
SELECT c.hadm_id,
  COUNT(b.hadm_id) AS prior_adm_12m,
  COUNT(b.hadm_id) FILTER (WHERE b.admittime >= c.admittime - INTERVAL 182 DAY) AS prior_adm_6m,
  COUNT(b.hadm_id) FILTER (WHERE b.edregtime IS NOT NULL AND b.admittime >= c.admittime - INTERVAL 182 DAY) AS prior_ed_adm_6m
FROM c LEFT JOIN adm b ON b.subject_id = c.subject_id AND b.admittime < c.admittime
     AND b.admittime >= c.admittime - INTERVAL 365 DAY
GROUP BY c.hadm_id''').df()
if find_table(ED, 'edstays') is not None:
    ed = con.execute(f'''
    SELECT c.hadm_id, COUNT(e.stay_id) AS prior_ed_6m
    FROM c LEFT JOIN {src('edstays', ED, types={'intime': TS, 'outtime': TS})} e
      ON e.subject_id = c.subject_id AND e.intime < c.admittime AND e.intime >= c.admittime - INTERVAL 182 DAY
      AND (e.hadm_id IS NULL OR e.hadm_id <> c.hadm_id)
    GROUP BY c.hadm_id''').df()
    ed_source = 'ed/edstays'
else:
    ed = util[['hadm_id']].assign(prior_ed_6m=util.prior_ed_adm_6m)
    ed_source = 'admisiones con paso por guardia (aproximación)'
print('Fuente de visitas a guardia:', ed_source)

dxc = con.execute('SELECT hadm_id, seq_num, icd_code, icd_version FROM dx WHERE hadm_id IN (SELECT hadm_id FROM c)').df()
flags = pd.concat([ru.comorbidity_flags(dxc.loc[dxc.icd_version == v, 'icd_code'], v) for v in (9, 10)]).sort_index()
flags['hadm_id'] = dxc.hadm_id
cflags = flags.groupby('hadm_id').max()
dx_feat = pd.DataFrame({'charlson': ru.charlson_score(cflags), 'n_diagnoses': dxc.groupby('hadm_id').size()})
dx_feat = dx_feat.join(cflags.add_prefix('cci_').astype(int))
principal = dxc.sort_values('seq_num').groupby('hadm_id').first()
dx_feat['dx1_group'] = [ru.icd9_group(c, dotless=True) if v == 9 else ru.icd10_group(c)
                        for c, v in zip(principal.icd_code, principal.icd_version)]
dx_feat['dx1_icd_version'] = principal.icd_version
oncology_dx = cflags[['cancer', 'mets']].any(axis=1)

proc = con.execute(f'''
SELECT hadm_id, COUNT(*) AS n_procedures FROM {src('procedures_icd', types={'icd_code': 'VARCHAR', 'icd_version': 'INTEGER', 'chartdate': 'DATE'})}
WHERE hadm_id IN (SELECT hadm_id FROM c) GROUP BY hadm_id''').df()
serv = con.execute(f'''
SELECT hadm_id, arg_max(curr_service, transfertime) AS discharge_service
FROM {src('services', types={'transfertime': TS})} WHERE hadm_id IN (SELECT hadm_id FROM c) GROUP BY hadm_id''').df()
ANTIDIAB = ('insulin|metformin|glipizide|glyburide|glimepiride|pioglitazone|rosiglitazone|sitagliptin|linagliptin|'
            'saxagliptin|alogliptin|liraglutide|exenatide|dulaglutide|semaglutide|empagliflozin|canagliflozin|'
            'dapagliflozin|ertugliflozin|repaglinide|nateglinide|acarbose|miglitol')
meds = con.execute(f'''
SELECT TRY_CAST(hadm_id AS BIGINT) AS hadm_id, COUNT(DISTINCT lower(drug)) AS n_drugs,
  COUNT(DISTINCT CASE WHEN regexp_matches(lower(drug), '{ANTIDIAB}') THEN lower(drug) END) AS n_antidiabetic,
  MAX(CASE WHEN lower(drug) LIKE '%insulin%' THEN 1 ELSE 0 END) AS insulin,
  MAX(CASE WHEN lower(drug) LIKE '%metformin%' THEN 1 ELSE 0 END) AS metformin
FROM {src('prescriptions', all_varchar=True)}
WHERE TRY_CAST(hadm_id AS BIGINT) IN (SELECT hadm_id FROM c) GROUP BY 1''').df()

In [ ]:
# itemid de MIMIC-IV hosp: hemoglobina 51222 (gases 50811), sodio 50983 (gases 50824),
# HbA1c 50852, glucosa 50931 (gases 50809).
LAB_ITEMS = {'hb': [51222, 50811], 'na': [50983, 50824], 'hba1c': [50852], 'glucose': [50931, 50809]}
lab_cache = CACHE / f'labs_{len(cohort)}.parquet'
if not lab_cache.exists():
    t0 = time.perf_counter()
    items = ', '.join(str(i) for v in LAB_ITEMS.values() for i in v)
    con.execute(f'''
    COPY (
      SELECT c.hadm_id, l.itemid, arg_max(l.valuenum, l.charttime) AS last_value, max(l.valuenum) AS max_value
      FROM {src('labevents', types={'charttime': TS, 'storetime': TS, 'valuenum': 'DOUBLE', 'value': 'VARCHAR',
                                     'ref_range_lower': 'DOUBLE', 'ref_range_upper': 'DOUBLE', 'hadm_id': 'BIGINT',
                                     'comments': 'VARCHAR', 'flag': 'VARCHAR', 'priority': 'VARCHAR'})} l
      JOIN c ON l.subject_id = c.subject_id AND l.charttime BETWEEN c.admittime AND c.dischtime
      WHERE l.itemid IN ({items}) AND l.valuenum IS NOT NULL
      GROUP BY c.hadm_id, l.itemid) TO '{lab_cache.as_posix()}' (FORMAT PARQUET)''')
    print(f'labevents leído en {time.perf_counter() - t0:.0f} s; caché en {lab_cache}')
labs_long = con.execute(f"SELECT * FROM read_parquet('{lab_cache.as_posix()}')").df()
labs = pd.DataFrame(index=pd.Index(cohort.hadm_id.unique(), name='hadm_id'))
for name, ids in LAB_ITEMS.items():
    sub = labs_long[labs_long.itemid.isin(ids)].copy()
    sub['prio'] = sub.itemid.map({i: k for k, i in enumerate(ids)})
    sub = sub.sort_values('prio').groupby('hadm_id').first()
    labs[f'{name}_last'] = sub.last_value
    labs[f'{name}_max'] = sub.max_value
labs['hba1c_cat'] = pd.cut(labs.hba1c_max, [-np.inf, 7, 8, np.inf], right=False, labels=['Norm', '>7', '>8']).astype(object)
labs['hba1c_cat'] = labs.hba1c_cat.fillna('None')
labs['glucose_cat'] = pd.cut(labs.glucose_max, [-np.inf, 200, 300, np.inf], right=False, labels=['Norm', '>200', '>300']).astype(object)
labs['glucose_cat'] = labs.glucose_cat.fillna('None')

df = (cohort.merge(util, on='hadm_id', how='left').merge(ed, on='hadm_id', how='left')
      .merge(dx_feat, left_on='hadm_id', right_index=True, how='left').merge(proc, on='hadm_id', how='left')
      .merge(serv, on='hadm_id', how='left').merge(meds, on='hadm_id', how='left')
      .merge(labs[['hb_last', 'na_last', 'glucose_last', 'hba1c_cat', 'glucose_cat']], left_on='hadm_id',
             right_index=True, how='left'))
for c in ['n_procedures', 'n_drugs', 'n_antidiabetic', 'insulin', 'metformin', 'prior_ed_6m']:
    df[c] = df[c].fillna(0)
df['oncology'] = df.discharge_service.eq('OMED') | df.hadm_id.map(oncology_dx).fillna(False)
EMERGENT = ['EW EMER.', 'DIRECT EMER.', 'URGENT']
df['emergent'] = df.admission_type.isin(EMERGENT) | df.admission_location.eq('EMERGENCY ROOM')
df['LACE'] = ru.lace_score(df.los_days, df.emergent, df.charlson.fillna(0), df.prior_ed_6m)
df['HOSPITAL'] = ru.hospital_score(df.hb_last < 12, df.oncology, df.na_last < 135, df.n_procedures > 0,
                                   ~df.admission_type.isin(ELECTIVE), df.prior_adm_12m, df.los_days)
NUM = ['age', 'los_days', 'prior_adm_6m', 'prior_adm_12m', 'prior_ed_6m', 'n_diagnoses', 'charlson',
       'n_procedures', 'hb_last', 'na_last', 'glucose_last', 'n_drugs', 'n_antidiabetic', 'insulin', 'metformin'] + \
      [c for c in df if c.startswith('cci_')]
CAT = ['gender', 'admission_type', 'admission_location', 'discharge_location', 'discharge_service',
       'dx1_group', 'hba1c_cat', 'glucose_cat']
X = df[NUM + CAT].copy()
for c in CAT:
    X[c] = X[c].astype(object).where(X[c].notna(), None)
y, groups = df.y, df.subject_id
print(f'{X.shape[1]} features ({len(NUM)} numéricas, {len(CAT)} categóricas)')
display(X.isna().mean().sort_values(ascending=False).head(8).rename('faltante').to_frame().round(3))

## 3. Partición temporal
Se verifica que ningún paciente cruce particiones y se muestra el año real estimado de las admisiones de cada partición. Ese año es aproximado: MIMIC desplaza las fechas de cada paciente y solo informa el trienio real de su `anchor_year`.

In [ ]:
SPLIT = {'2008-2010': 'train', '2011-2013': 'train', '2014-2016': 'validation', '2017-2019': 'oot',
         '2020-2022': 'oot2'}
df['partition'] = df.period.map(SPLIT)
assert df.partition.notna().all(), f'Períodos no previstos: {set(df.period) - set(SPLIT)}'
df['est_year'] = df.period.str[:4].astype(int) + 1 + (df.admittime.dt.year - df.anchor_year)
idx = {k: np.flatnonzero(df.partition.eq(k)) for k in ['train', 'validation', 'oot', 'oot2'] if df.partition.eq(k).any()}
parts = list(idx)
for i, a in enumerate(parts):
    for b in parts[i+1:]:
        assert set(groups.iloc[idx[a]]).isdisjoint(groups.iloc[idx[b]])
print('Pacientes compartidos entre particiones: 0')
split_table = df.groupby(['partition', 'period']).agg(episodios=('y', 'size'), pacientes=('subject_id', 'nunique'),
    prevalencia=('y', 'mean'), cie10_principal=('dx1_icd_version', lambda s: (s == 10).mean()),
    año_est_p05=('est_year', lambda s: s.quantile(.05)), año_est_p95=('est_year', lambda s: s.quantile(.95)))
display(split_table.round(3))

## 4. Baselines y validación cruzada temporal
LACE y HOSPITAL completos se convierten en probabilidad con una logística ajustada en train. Luego se corre la misma grilla en dos etapas que en UCI sobre los dos folds de ventana expansiva. Con solo dos folds, el error estándar de la regla de selección es una aproximación gruesa.

In [ ]:
tr_i, va_i = idx['train'], idx['validation']
Xtr, ytr, Xva, yva = X.iloc[tr_i], y.iloc[tr_i], X.iloc[va_i], y.iloc[va_i]
score_models = {s: LogisticRegression().fit(df.iloc[tr_i][[s]], ytr) for s in ['LACE', 'HOSPITAL']}
def score_prob(s, ix):
    return score_models[s].predict_proba(df.iloc[ix][[s]])[:, 1]

dev_i = np.concatenate([tr_i, va_i])
Xdev, ydev = X.iloc[dev_i].reset_index(drop=True), y.iloc[dev_i].reset_index(drop=True)
pdev = df.period.iloc[dev_i].reset_index(drop=True)
folds = [('2008-10 → 2011-13', np.flatnonzero(pdev.eq('2008-2010')), np.flatnonzero(pdev.eq('2011-2013'))),
         ('2008-13 → 2014-16', np.flatnonzero(pdev.isin(['2008-2010', '2011-2013'])), np.flatnonzero(pdev.eq('2014-2016')))]
folds = [f for f in folds if len(f[1]) and len(f[2])]
XGB_GRID = [dict(max_depth=d, colsample_bytree=c) for d in (3, 4) for c in (0.5, 0.8)]
LGBM_GRID = [dict(num_leaves=l, colsample_bytree=c) for l in (7, 15) for c in (0.5, 0.8)]
stage1 = ru.build_grid(XGB_GRID, LGBM_GRID, strategies=('none',))
t0 = time.perf_counter()
cv1 = ru.evaluate_folds(Xdev, ydev, folds, stage1, NUM, CAT, seed=SEED, n_jobs=N_JOBS)
sum1 = ru.summarize_cv(cv1, stage1)
best_configs = set(sum1.loc[sum1.groupby('kind').AP_mean.idxmax(), 'config'])
stage2 = [c for c in ru.build_grid(XGB_GRID, LGBM_GRID) if c['config'] in best_configs and c['strategy'] != 'none']
cv2 = ru.evaluate_folds(Xdev, ydev, folds, stage2, NUM, CAT, seed=SEED, n_jobs=N_JOBS)
print(f'CV temporal: {len(stage1) + len(stage2)} combinaciones × {len(folds)} folds en {time.perf_counter() - t0:.0f} s')
combos = stage1 + stage2
cv_all = pd.concat([cv1, cv2], ignore_index=True)
summary = ru.summarize_cv(cv_all, combos)
chosen_name, best_name, se_best, candidates = ru.select_one_se(summary)
chosen = next(c for c in combos if c['name'] == chosen_name)
display(summary[['kind', 'strategy', 'AP_mean', 'AP_std', 'AUROC_mean', 'Brier_mean', 'Lift@20_mean']].round(4))
display(cv_all.pivot_table(index='combo', columns='fold', values='AP').loc[summary.index].round(4))
print('MODELO ELEGIDO (regla de 1 EE):', chosen_name)
ru.mlflow_log_summary(summary, 'reingresos_mimic_temporal_cv', MLRUNS,
                      common_params={'cv': 'expanding window by anchor_year_group', 'seed': SEED},
                      tags={'dataset': 'MIMIC-IV'})
summary.to_csv(OUT / 'cv_temporal_resumen.csv', index_label='combinacion')

## 5. Modelo final: train → recalibración en validación → OOT

In [ ]:
pos_weight = float((ytr == 0).sum() / (ytr == 1).sum())
best = ru.make_pipeline(chosen['kind'], chosen['strategy'], NUM, CAT, chosen['params'], pos_weight, SEED).fit(Xtr, ytr)
cal_method, cal_table, cal_oof = ru.choose_calibration(best, Xva, yva, groups.iloc[va_i])
display(cal_table.round(5))
final = ru.fit_calibrator(best, Xva, yva, cal_method)
p20 = top_metrics(yva, cal_oof[cal_method], .20)['threshold']
print(f'Recalibración: {cal_method}. Umbral equivalente al cupo 20 % en validación: {p20:.3f}')

rows = []
for part, ix in idx.items():
    yy = y.iloc[ix]
    if yy.nunique() < 2:
        continue
    p = final.predict_proba(X.iloc[ix])[:, 1]
    for name, pp in [('Modelo', p), ('LACE', score_prob('LACE', ix)), ('HOSPITAL', score_prob('HOSPITAL', ix))]:
        rows.append({'partición': part, 'modelo': name, 'n': len(ix), 'prevalencia': yy.mean(),
                     'pred_media': pp.mean(), **metrics(yy, pp), 'fracción_≥umbral': (pp >= p20).mean()})
degradation = pd.DataFrame(rows)
print('Desempeño del modelo congelado por partición (train es dentro de muestra):')
display(degradation.round(4))
by_period = []
for per in sorted(df.period.unique()):
    ix = np.flatnonzero(df.period.eq(per))
    if y.iloc[ix].nunique() == 2:
        p = final.predict_proba(X.iloc[ix])[:, 1]
        by_period.append({'período': per, 'partición': SPLIT[per], 'n': len(ix), 'prevalencia': y.iloc[ix].mean(),
                          'pred_media': p.mean(), **metrics(y.iloc[ix], p)})
by_period = pd.DataFrame(by_period)
display(by_period.round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(by_period['período'], by_period.AUROC, 'o-', label='AUROC')
axes[0].set(title='Discriminación por período', ylabel='AUROC')
ax2 = axes[0].twinx(); ax2.plot(by_period['período'], by_period.AP, 's--', color='#D36B36', label='AP'); ax2.set_ylabel('AP')
axes[1].plot(by_period['período'], by_period.prevalencia, 'o-', label='Observada')
axes[1].plot(by_period['período'], by_period.pred_media, 's--', label='Predicha media')
axes[1].set(title='Calibración global por período', ylabel='Reingreso <30 días'); axes[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / 'degradacion_temporal.png', dpi=170); plt.show()

In [ ]:
oot = idx['oot']
yo = y.iloc[oot]
po = final.predict_proba(X.iloc[oot])[:, 1]
probs = {'Modelo': po, 'LACE': score_prob('LACE', oot), 'HOSPITAL': score_prob('HOSPITAL', oot)}
boot = ru.grouped_bootstrap(yo, probs, groups.iloc[oot], n_boot=300, seed=SEED + 2,
                            pairs=[('Modelo', 'LACE'), ('Modelo', 'HOSPITAL')])
ci_all = ru.bootstrap_ci(boot)
display(pd.DataFrame({n: metrics(yo, p) for n, p in probs.items()}).T.round(4))
display(ci_all.round(4))
sens = {'Reingreso cualquier causa': metrics(yo, po), 'Sin reingresos electivos': metrics(df.y_no_elective.iloc[oot], po)}
print('Sensibilidad del outcome (OOT):'); display(pd.DataFrame(sens).T.round(4))
thresholds = np.round(np.arange(0.01, 0.41, 0.01), 2)
dca_oot = dcurves.dca(data=pd.DataFrame({'y': yo.values, **probs}), outcome='y', modelnames=list(probs),
                      thresholds=thresholds)
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
for n, p in probs.items():
    pr, rc, _ = precision_recall_curve(yo, p); axes[0].plot(rc, pr, label=f'{n} (AP={average_precision_score(yo, p):.3f})')
axes[0].axhline(yo.mean(), ls='--', color='gray'); axes[0].set(xlabel='Recall', ylabel='Precision', title='Precision–recall (OOT)', ylim=(0, 1))
axes[0].legend(fontsize=8)
fr, mp = calibration_curve(yo, po, n_bins=10, strategy='quantile')
axes[1].plot(mp, fr, 'o-'); axes[1].plot([0, mp.max()], [0, mp.max()], '--', color='gray')
axes[1].set(xlabel='Predicha', ylabel='Observada', title='Calibración (OOT)')
for n, g in dca_oot.groupby('model', sort=False):
    axes[2].plot(g.threshold, g.net_benefit, label={'all': 'Intervenir a todos', 'none': 'No intervenir'}.get(n, n),
                 ls='--' if n in ('all', 'none') else '-')
axes[2].axvline(p20, color='#D36B36', ls=':', label='Cupo 20 % (validación)')
axes[2].set(ylim=(-0.01, yo.mean() * 1.05), xlabel='Umbral', ylabel='Beneficio neto', title='Curva de decisión (OOT)')
axes[2].legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / 'evaluacion_oot.png', dpi=170); plt.show()

## 6. Subgrupos en OOT y exportación

In [ ]:
ct = df.iloc[oot].reset_index(drop=True)
ct['age_group'] = pd.cut(ct.age, [17, 44, 64, 79, 200], labels=['18-44', '45-64', '65-79', '80+']).astype(str)
ya, ga = yo.to_numpy(), groups.iloc[oot].to_numpy()
subgroups = []
for col in ['gender', 'age_group', 'hba1c_cat', 'dx1_icd_version']:
    for v in sorted(ct[col].astype(str).unique()):
        m = ct[col].astype(str).eq(v).to_numpy()
        if m.sum() < 200 or ya[m].sum() < 20:
            continue
        b = ru.grouped_bootstrap(ya[m], {'m': po[m]}, ga[m], n_boot=200, seed=SEED + 3)
        q = b[b.metrica == 'AUROC'].valor.quantile([.025, .975])
        subgroups.append({'variable': col, 'grupo': v, 'n': int(m.sum()), 'prevalencia': ya[m].mean(),
                          'AUROC': roc_auc_score(ya[m], po[m]), 'AUROC_lo': q.iloc[0], 'AUROC_hi': q.iloc[1],
                          'AP': average_precision_score(ya[m], po[m])})
subgroups = pd.DataFrame(subgroups)
display(subgroups.round(3))
result = {'fecha': '2026-10-08', 'versions': versions, 'mimic_dir': str(MIMIC_DIR), 'ed_source': ed_source,
          'cohort_flow': flow, 'split': split_table.reset_index().to_dict('records'),
          'features': NUM + CAT, 'cv_summary': summary[['kind', 'strategy', 'AP_mean', 'AP_std', 'AUROC_mean']].reset_index().to_dict('records'),
          'chosen': chosen_name, 'calibration': cal_method, 'threshold_top20_validation': p20,
          'degradation': degradation.to_dict('records'), 'by_period': by_period.to_dict('records'),
          'oot_ci95': ci_all.reset_index().to_dict('records'), 'outcome_sensitivity': sens,
          'subgroups': subgroups.to_dict('records')}
(OUT / 'results_mimic.json').write_text(json.dumps(result, indent=2, ensure_ascii=False, default=str), encoding='utf-8')
degradation.to_csv(OUT / 'degradacion_temporal.csv', index=False)
by_period.to_csv(OUT / 'desempeno_por_periodo.csv', index=False)
print('Resultados en', OUT)

## 7. Interpretación y próximos pasos
- **Qué responder con este notebook:** cuánto cae AUROC/AP y cuánto se descalibra el modelo entre validación (2014–16) y OOT (2017–19, 2020–22), y si el modelo sigue superando a LACE y HOSPITAL completos fuera de tiempo. Si la calibración global se desplaza pero la discriminación se mantiene, alcanza con recalibrar periódicamente; si cae la discriminación, hay que reentrenar.
- **Transición CIE-9 → CIE-10:** la tabla de partición muestra la fracción de diagnósticos principales en CIE-10 por período; el subgrupo por versión de codificación en OOT ayuda a detectar si el cambio afecta al modelo.
- **Límites:** un solo centro académico; el reingreso se observa solo dentro del BIDMC; año real aproximado por `anchor_year_group`; no se modelan las muertes posalta como riesgo competitivo.
- **Extensión con texto clínico:** con acceso a MIMIC-IV-Note (credencial adicional), representar el resumen de alta con un modelo clínico de lenguaje (Bio_ClinicalBERT u otro de contexto largo) y medir la ganancia sobre las variables tabulares con la misma partición temporal.